# CXR-Recall reliability, step 0: is there anything for a reliability gate to gain?

**Question.** D9 showed that, on average, retrieval (top-5 RAD-DINO neighbours) is no better than random neighbours. A reliability-aware system that decides per query whether to trust its neighbours can only help if retrieval **helps some queries and hurts others**. This notebook measures whether that is the case before anything new is built.

**What it does.** Exactly the D9 run (same split, frozen RAD-DINO, `probe` / `retrieval` / `random_control`, LR sweep on val, 5 seeds), plus:
1. Saves every arm's per-query val and test probabilities, for every seed.
2. Computes per-query reliability features from the retrieved neighbours: similarity (top-1, mean, gap), agreement among the neighbours' labels, agreement between the neighbours and the probe's prediction, and the probe's own uncertainty. All are available at inference time.
3. **Heterogeneity:** per-query gain of retrieval over the probe (and over random neighbours), by reliability band. Bands are set on val; reported on test.
4. **Upper bound:** an oracle that picks the better model per query using the true labels. If even this gains little, stop.
5. **A realistic gate:** route each query to the probe or the retrieval model by one feature, with threshold and direction chosen on val. Compared against simply averaging the two models, so a gain from ensembling isn't mistaken for a gain from reliability.
6. **Calibration** (ECE, Brier) per arm.

**Settings:** GPU on, Internet on, attach `raddar/chest-xrays-indiana-university`. **Save Version, Save & Run All.** About 20-30 min on a T4.

**What to paste back:** the final `RESULT_JSON:` line. Also download `reliability_step0_outputs.npz` from the output tab for local analysis.

In [ ]:
import os, glob, json, time, copy, random, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from scipy import stats

print('torch:', torch.__version__, ' cuda available:', torch.cuda.is_available())

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 1. Configuration

In [ ]:
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working'
RADDINO_ID = 'microsoft/rad-dino'   # pretrained on MIMIC-CXR, CheXpert, NIH, PadChest, BRAX -- not Open-I
EXTRACT_BATCH = 32
K = 5
SEEDS = [0, 1, 2, 3, 4]
LRS = [1e-4, 3e-4, 1e-3]            # swept per arm and seed, chosen on val
MAX_EPOCHS, PATIENCE = 200, 15
BATCH_SIZE = 64
ARMS = ['probe', 'retrieval', 'random_control']
N_BOOT = 2000
N_BANDS = 5
# ViT-based system, 3 seeds (ARCHITECTURE.md section 4), for context only -- not paired with this run
VIT_REFERENCE = {'baseline_test': 0.7131, 'cxr_recall_test': 0.7090, 'margin_test': -0.0042}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Locate the attached dataset (identical to the multi-seed notebook)

In [ ]:
import os

candidates = os.listdir(INPUT_DIR)
assert candidates, "No dataset attached — use '+ Add Data' and search for the Indiana University chest x-ray collection first."
print('attached datasets:', candidates)

DATASET_DIR = os.path.join(INPUT_DIR, candidates[0])
print('using:', DATASET_DIR)
for root, dirs, files in os.walk(DATASET_DIR):
    print(root, '->', dirs, files[:5])

In [ ]:
import glob

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
png_dirs = sorted(set(os.path.dirname(p) for p in glob.glob(os.path.join(DATASET_DIR, '**', '*.png'), recursive=True)))
print('csv files:', csv_files)
print('folders containing pngs:', png_dirs)

## 3. Manifest, label vocabulary, fixed study-level split (identical to the multi-seed notebook)

In [ ]:
reports_csv = [f for f in csv_files if 'report' in os.path.basename(f).lower()]
projections_csv = [f for f in csv_files if 'projection' in os.path.basename(f).lower()]
assert reports_csv and projections_csv, 'Could not auto-find reports/projections CSVs -- inspect csv_files above and set paths manually.'

reports_df = pd.read_csv(reports_csv[0])
proj_df = pd.read_csv(projections_csv[0])

UID_COL, FINDINGS_COL, IMPRESSION_COL, TAGS_COL = 'uid', 'findings', 'impression', 'Problems'
PROJ_UID_COL, FILENAME_COL = 'uid', 'filename'

merged = proj_df.merge(reports_df, left_on=PROJ_UID_COL, right_on=UID_COL, suffixes=('', '_report'))

def build_report_text(row):
    parts = [str(row.get(FINDINGS_COL, '')), str(row.get(IMPRESSION_COL, ''))]
    return ' '.join(p for p in parts if p and p.lower() != 'nan').strip()

def build_tags(row):
    raw = row.get(TAGS_COL, '')
    if pd.isna(raw):
        return ''
    sep = ';' if ';' in str(raw) else ','
    return ';'.join(t.strip() for t in str(raw).split(sep) if t.strip())

manifest = pd.DataFrame({
    'image_path': merged[FILENAME_COL],
    'patient_id': merged[UID_COL].astype(str),
    'report_text': merged.apply(build_report_text, axis=1),
    'mesh_tags': merged.apply(build_tags, axis=1),
})

IMAGE_ROOT = png_dirs[0]
print('manifest rows:', len(manifest), ' image root:', IMAGE_ROOT)
manifest.head()

In [ ]:
from collections import Counter
import numpy as np

def build_mesh_vocab(tag_lists, top_k=14):
    counts = Counter(tag for tags in tag_lists for tag in tags)
    return [tag for tag, _ in counts.most_common(top_k)]

def mesh_tags_to_vector(tags, vocab):
    tag_set = set(tags)
    return np.array([1.0 if v in tag_set else 0.0 for v in vocab], dtype=np.float32)

In [ ]:
def patient_level_split(df, patient_id_col, ratios=(0.8, 0.1, 0.1), seed=42):
    patients = sorted(df[patient_id_col].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train, n_val = int(n * ratios[0]), int(n * ratios[1])
    train_p = set(patients[:n_train])
    val_p = set(patients[n_train:n_train + n_val])
    test_p = set(patients[n_train + n_val:])
    return (df[df[patient_id_col].isin(train_p)],
            df[df[patient_id_col].isin(val_p)],
            df[df[patient_id_col].isin(test_p)])

train_df, val_df, test_df = patient_level_split(manifest, 'patient_id')
print('split sizes:', len(train_df), len(val_df), len(test_df))

train_tags = [t.split(';') if t else [] for t in train_df['mesh_tags']]
label_vocab = build_mesh_vocab(train_tags, top_k=14)
print('label vocab:', label_vocab)

EXPECTED_VOCAB = ['normal', 'Lung', 'Opacity', 'Cardiomegaly', 'Pulmonary Atelectasis', 'Calcinosis',
                  'Calcified Granuloma', 'Thoracic Vertebrae', 'Cicatrix', 'Spine', 'Pleural Effusion',
                  'Aorta', 'Markings', 'Diaphragm']
if INPUT_DIR == '/kaggle/input':
    assert (len(train_df), len(val_df), len(test_df)) == (5977, 746, 743), 'split differs from every other Phase 1 run'
    assert label_vocab == EXPECTED_VOCAB, 'label vocabulary differs from every other Phase 1 run'
    print('split and vocabulary match the documented Phase 1 runs')

## 4. RAD-DINO [CLS] embeddings (frozen, extracted once)

In [ ]:
def tags_to_vec(tag_str):
    tags = tag_str.split(';') if tag_str else []
    return mesh_tags_to_vector(tags, label_vocab)

class PathLabelDataset(Dataset):
    def __init__(self, df, preprocess):
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return self.preprocess(os.path.join(IMAGE_ROOT, row['image_path'])), tags_to_vec(row['mesh_tags'])

def make_raddino():
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained(RADDINO_ID)
    model = AutoModel.from_pretrained(RADDINO_ID).to(device).eval()
    def preprocess(path):
        return processor(images=Image.open(path).convert('RGB'), return_tensors='pt')['pixel_values'][0]
    return model, preprocess

@torch.no_grad()
def extract(model, preprocess, df):
    loader = DataLoader(PathLabelDataset(df, preprocess), batch_size=EXTRACT_BATCH, shuffle=False, num_workers=2)
    feats, labels = [], []
    for x, y in loader:
        feats.append(model(pixel_values=x.to(device)).pooler_output.float().cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

split_dfs = {'train': train_df, 'val': val_df, 'test': test_df}
EMB_PATH = os.path.join(WORK_DIR, 'raddino_cls_embeddings.npz')
t0 = time.time()
raddino, preprocess = make_raddino()
EMB, LBL = {}, {}
for s, df in split_dfs.items():
    EMB[s], LBL[s] = extract(raddino, preprocess, df)
del raddino; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
PIDS = {s: df['patient_id'].astype(str).values for s, df in split_dfs.items()}
np.savez_compressed(EMB_PATH, **{f'{s}_emb': EMB[s] for s in EMB}, **{f'{s}_lbl': LBL[s] for s in LBL},
                    **{f'{s}_pid': PIDS[s] for s in PIDS})
print({s: EMB[s].shape for s in EMB}, f'extracted in {(time.time() - t0) / 60:.1f} min, saved to {EMB_PATH}')

## 5. Neighbours

`retrieval`: cosine top-5 among train images. Train queries skip every image from their own study. `random_control`: 5 random train images per query (also never from the query's own study), redrawn per seed and fixed for that seed's training.

In [ ]:
def l2n(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

TRAIN_N = torch.from_numpy(l2n(EMB['train'])).to(device)

def topk_train(q, k):
    out = []
    for i in range(0, len(q), 1024):
        sims = torch.from_numpy(l2n(q[i:i + 1024])).to(device) @ TRAIN_N.T
        out.append(torch.topk(sims, k, dim=1).indices.cpu().numpy())
    return np.concatenate(out)

train_pids = PIDS['train']
MAX_VIEWS = int(train_df.groupby('patient_id').size().max())
raw = topk_train(EMB['train'], K + MAX_VIEWS)
nb_train, same_study_hits = [], 0
for i, row in enumerate(raw):
    index_only = [j for j in row if j != i][:K]
    same_study_hits += int(any(train_pids[j] == train_pids[i] for j in index_only))
    keep = [j for j in row if train_pids[j] != train_pids[i]][:K]
    assert len(keep) == K, f'query {i}: fewer than {K} neighbours outside its study'
    nb_train.append(keep)
NB_RETR = {'train': np.array(nb_train), 'val': topk_train(EMB['val'], K), 'test': topk_train(EMB['test'], K)}
leakage_rate = same_study_hits / len(train_pids)
print(f'LEAKAGE CHECK: {same_study_hits}/{len(train_pids)} ({leakage_rate:.1%}) train queries would have retrieved '
      f'their own study without the exclusion (excluded below)')
assert not any(train_pids[j] == train_pids[i] for i, row in enumerate(NB_RETR['train']) for j in row)

def random_neighbours(seed):
    rng = np.random.default_rng(1000 + seed)
    out = {}
    for s in split_dfs:
        q_pids = PIDS[s]
        nb = np.empty((len(q_pids), K), dtype=int)
        for i in range(len(q_pids)):
            while True:
                c = rng.choice(len(train_pids), K, replace=False)
                if not np.any(train_pids[c] == q_pids[i]):
                    break
            nb[i] = c
        out[s] = nb
    return out

def p_at_k(nb, q_lbl):
    return float(((LBL['train'][nb] * q_lbl[:, None, :]).sum(-1) > 0).mean())

_rnd = random_neighbours(0)
for s in ('val', 'test'):
    print(f'{s}: retrieval P@{K} = {p_at_k(NB_RETR[s], LBL[s]):.4f}   random P@{K} = {p_at_k(_rnd[s], LBL[s]):.4f}')
print('(encoder comparison, raddino_cls: val 0.3252, test 0.3211)')

## 5b. Per-query reliability features (from the retrieved neighbours; all available at inference)

- `top1_sim`, `mean_sim`, `sim_gap` (top-1 minus 5th): how close the neighbours are.
- `nb_consensus`: mean pairwise Jaccard between the 5 neighbours' label sets (do the neighbours agree with each other?).
- `probe_nb_agree` and `probe_entropy` are added after training, from the probe's own predictions.

In [ ]:
@torch.no_grad()
def neighbour_sims(split):
    q = torch.from_numpy(l2n(EMB[split])).to(device)
    nb = torch.as_tensor(NB_RETR[split], device=device)
    return (q[:, None, :] * TRAIN_N[nb]).sum(-1).cpu().numpy()   # (n, K), sorted descending for val/test

def pairwise_jaccard(nl):
    # nl: (n, K, L) neighbour label sets -> mean Jaccard over the K*(K-1)/2 pairs; two empty sets count as 1
    vals = []
    for a in range(K):
        for b in range(a + 1, K):
            inter = (nl[:, a] * nl[:, b]).sum(-1)
            union = ((nl[:, a] + nl[:, b]) > 0).sum(-1)
            vals.append(np.where(union > 0, inter / np.maximum(union, 1), 1.0))
    return np.mean(vals, axis=0)

FEATS = {}
for s in ('val', 'test'):
    sims = neighbour_sims(s)
    FEATS[s] = {'top1_sim': sims[:, 0], 'mean_sim': sims.mean(1), 'sim_gap': sims[:, 0] - sims[:, -1],
                'nb_consensus': pairwise_jaccard(LBL['train'][NB_RETR[s]])}
    print(s, {k: round(float(np.median(v)), 4) for k, v in FEATS[s].items()}, '(medians)')

## 6. Models (EvidenceEncoder copied verbatim from `src/models/evidence_encoder.py`)

In [ ]:
"""Evidence encoder + cross-attention fusion for Arm 3 (CXR-Recall). Takes
the query's own image embedding plus information about its k retrieved
neighbors (from Arm 2's FAISS index) and produces a fused representation
that the classification head consumes -- this is what makes retrieval
causally relevant to the diagnosis, unlike Arm 2's display-only retrieval.

Three evidence modes, in order of what was actually tried (see RESULTS.md):
  - label-only (Arm 3): each neighbor is represented purely by its 14-dim
    multi-hot label vector. Causally used but net *hurt* accuracy -- ~70% of
    retrieved neighbors don't share a label with the query, and label-only
    evidence gives the model no way to tell a strong visual match from a
    weak one before trusting it.
  - early fusion, label+image (Arm 3b): each neighbor's image embedding
    (already computed during retrieval) and label vector are concatenated
    and projected into ONE combined vector per neighbor *before* attention
    runs. Closed ~82% of Arm 3's gap to Arm 1. The image embedding is passed
    through its own learned `image_proj` rather than used raw, as a
    mitigation for the "shared representation" collapse risk noted in the
    KNN literature (sharing retrieval and classification representations
    directly can hurt performance).
  - joint attention, label+image (Arm 3c): image and label information for
    each neighbor are kept as SEPARATE attendable tokens (2k tokens total,
    not k pre-fused ones), tagged with a learned modality embedding (so the
    model knows which token is an image vs. a label) and a learned
    neighbor-position embedding (so the model knows which image/label pair
    belongs to the same neighbor). This lets the query attend to a specific
    neighbor's image independently of that neighbor's label, rather than
    having the two pre-blended by a fixed linear combination before
    attention ever runs -- a strictly more expressive mechanism than early
    fusion, at the cost of more attendable items (2k vs k) on a small
    dataset, which is worth watching for overfitting."""

import torch
import torch.nn as nn


class EvidenceEncoder(nn.Module):
    def __init__(self, num_labels: int, embed_dim: int, num_heads: int = 4,
                 use_image_embeddings: bool = False, image_embed_dim: int = None,
                 joint_attention: bool = False, max_k: int = 16):
        super().__init__()
        self.use_image_embeddings = use_image_embeddings
        self.joint_attention = joint_attention
        self.label_proj = nn.Linear(num_labels, embed_dim)

        if use_image_embeddings:
            image_embed_dim = image_embed_dim or embed_dim
            self.image_proj = nn.Linear(image_embed_dim, embed_dim)
            if joint_attention:
                self.modality_embed = nn.Embedding(2, embed_dim)  # 0 = image, 1 = label
                self.neighbor_pos_embed = nn.Embedding(max_k, embed_dim)
            else:
                self.combine = nn.Linear(embed_dim * 2, embed_dim)
        elif joint_attention:
            raise ValueError("joint_attention=True requires use_image_embeddings=True")

        self.attention = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self, query_embedding: torch.Tensor, evidence_labels: torch.Tensor,
                evidence_image_embeddings: torch.Tensor = None) -> torch.Tensor:
        """query_embedding: (batch, embed_dim). evidence_labels: (batch, k, num_labels).
        evidence_image_embeddings: (batch, k, image_embed_dim), required iff
        use_image_embeddings=True -- the retrieved neighbors' own image
        embeddings, precomputed outside this module during retrieval."""
        label_embed = self.label_proj(evidence_labels)  # (batch, k, embed_dim)

        if self.use_image_embeddings:
            if evidence_image_embeddings is None:
                raise ValueError("use_image_embeddings=True requires evidence_image_embeddings")
            image_embed = self.image_proj(evidence_image_embeddings)  # (batch, k, embed_dim)

            if self.joint_attention:
                k = evidence_labels.shape[1]
                device = evidence_labels.device
                pos = self.neighbor_pos_embed(torch.arange(k, device=device)).unsqueeze(0)  # (1, k, embed_dim)
                image_type = self.modality_embed.weight[0]  # (embed_dim,)
                label_type = self.modality_embed.weight[1]  # (embed_dim,)
                image_tok = image_embed + pos + image_type
                label_tok = label_embed + pos + label_type
                evidence_embed = torch.cat([image_tok, label_tok], dim=1)  # (batch, 2k, embed_dim) -- separately attendable
            else:
                evidence_embed = self.combine(torch.cat([image_embed, label_embed], dim=-1))  # (batch, k, embed_dim)
        else:
            evidence_embed = label_embed

        query = query_embedding.unsqueeze(1)  # (batch, 1, embed_dim)
        context, _weights = self.attention(query, evidence_embed, evidence_embed)
        return context.squeeze(1)  # (batch, embed_dim)

In [ ]:
D = EMB['train'].shape[1]
MU, SD = EMB['train'].mean(0), EMB['train'].std(0) + 1e-6
X = {s: torch.from_numpy((EMB[s] - MU) / SD).float().to(device) for s in EMB}   # standardized with train stats
Y = {s: torch.from_numpy(LBL[s]).float().to(device) for s in LBL}
L = len(label_vocab)

class Probe(nn.Module):
    """No retrieval: linear head on the frozen query embedding (as Arm 1's head)."""
    def __init__(self):
        super().__init__()
        self.head = nn.Linear(D, L)
    def forward(self, q, ev_lbl=None, ev_img=None):
        return self.head(q)

class Fusion(nn.Module):
    """Arm 3c-frozen on cached embeddings: joint attention over image/label tokens, residual fusion, linear head."""
    def __init__(self):
        super().__init__()
        self.evidence_encoder = EvidenceEncoder(L, D, use_image_embeddings=True, image_embed_dim=D,
                                                joint_attention=True, max_k=K)
        self.head = nn.Linear(D, L)
    def forward(self, q, ev_lbl, ev_img):
        return self.head(q + self.evidence_encoder(q, ev_lbl, ev_img))

def macro_auroc(y_true, y_score):
    scores = [roc_auc_score(y_true[:, i], y_score[:, i]) for i in range(y_true.shape[1])
              if len(np.unique(y_true[:, i])) == 2]
    return float(np.mean(scores)) if scores else float('nan')

def logits(model, split, idx, nb, zero=None):
    q = X[split][idx]
    if isinstance(model, Probe):
        return model(q)
    n = nb[split][idx]
    ev_img, ev_lbl = X['train'][n], Y['train'][n]
    if zero in ('all', 'img'):
        ev_img = torch.zeros_like(ev_img)
    if zero == 'all':
        ev_lbl = torch.zeros_like(ev_lbl)
    return model(q, ev_lbl, ev_img)

@torch.no_grad()
def probs(model, split, nb, zero=None):
    model.eval()
    idx = torch.arange(len(Y[split]), device=device)
    return torch.sigmoid(logits(model, split, idx, nb, zero)).cpu().numpy()

def train_one(arm, seed, lr, nb):
    set_seed(seed)
    model = (Probe() if arm == 'probe' else Fusion()).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(seed)
    n = len(Y['train'])
    best, best_state, best_epoch, bad, epoch = -1.0, None, 0, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, generator=gen).to(device)
        for b in range(0, n, BATCH_SIZE):
            idx = perm[b:b + BATCH_SIZE]
            opt.zero_grad()
            loss = crit(logits(model, 'train', idx, nb), Y['train'][idx])
            loss.backward()
            opt.step()
        val = macro_auroc(LBL['val'], probs(model, 'val', nb))
        if val > best:
            best, best_state, best_epoch, bad = val, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    hit_cap = bad < PATIENCE   # loop ran out of epochs before early stopping triggered
    model.load_state_dict(best_state)
    return model, {'val': best, 'epoch': best_epoch, 'stopped': epoch, 'hit_cap': hit_cap, 'lr': lr}

print('models defined; embedding dim', D, ' labels', L)

## 7. Train every arm, every seed

In [ ]:
results = []
PROBS = {arm: {} for arm in ARMS}   # PROBS[arm][seed][split] -> (n, L)
t_all = time.time()
for seed in SEEDS:
    t0 = time.time()
    nbs = {'probe': NB_RETR, 'retrieval': NB_RETR, 'random_control': random_neighbours(seed)}
    nbs = {a: {s: torch.as_tensor(v, device=device) for s, v in d.items()} for a, d in nbs.items()}  # index on the GPU
    r = {'seed': seed}
    for arm in ARMS:
        best_model, best_info = None, None
        for lr in LRS:
            model, info = train_one(arm, seed, lr, nbs[arm])
            print(f'[seed {seed} | {arm:14s}] lr={lr:.0e}  best val={info["val"]:.4f} (epoch {info["epoch"]}, '
                  f'stopped {info["stopped"]}{", HIT CAP" if info["hit_cap"] else ""})')
            if best_info is None or info['val'] > best_info['val']:
                best_model, best_info = model, info
        PROBS[arm][seed] = {s: probs(best_model, s, nbs[arm]) for s in ('val', 'test')}
        test = macro_auroc(LBL['test'], PROBS[arm][seed]['test'])
        r.update({f'{arm}_val': round(best_info['val'], 4), f'{arm}_test': round(test, 4),
                  f'{arm}_lr': best_info['lr'], f'{arm}_epoch': best_info['epoch'], f'{arm}_hit_cap': best_info['hit_cap']})
        if arm != 'probe':   # evidence-ablation shift on val, same measurement as every prior run
            real = probs(best_model, 'val', nbs[arm])
            r[f'{arm}_shift_full'] = round(float(np.abs(real - probs(best_model, 'val', nbs[arm], 'all')).mean()), 4)
            r[f'{arm}_shift_img'] = round(float(np.abs(real - probs(best_model, 'val', nbs[arm], 'img')).mean()), 4)
        print(f'[seed {seed} | {arm:14s}] chosen lr={best_info["lr"]:.0e}  val={best_info["val"]:.4f}  TEST={test:.4f}')
    r['margin_retr_vs_probe'] = round(r['retrieval_test'] - r['probe_test'], 4)
    r['margin_retr_vs_random'] = round(r['retrieval_test'] - r['random_control_test'], 4)
    r['margin_random_vs_probe'] = round(r['random_control_test'] - r['probe_test'], 4)
    r['minutes'] = round((time.time() - t0) / 60, 1)
    results.append(r)
    print(f'RESULT_SEED_JSON: {json.dumps(r)}\n')
print(f'all seeds done in {(time.time() - t_all) / 60:.1f} min')

## 8. Aggregate (should reproduce D9: probe ~0.746, retrieval ~0.753, random ~0.751)

In [ ]:
n = len(results)
print(f'=== D9: RAD-DINO frozen encoder, {n} seeds, TEST macro AUROC ===')
print('seed |  probe   retrieval  random_ctrl | retr-probe  retr-random  random-probe | img share (retr)')
for r in results:
    share = r['retrieval_shift_img'] / r['retrieval_shift_full'] if r['retrieval_shift_full'] else float('nan')
    print(f"{r['seed']:>4} | {r['probe_test']:.4f}   {r['retrieval_test']:.4f}    {r['random_control_test']:.4f}    | "
          f"{r['margin_retr_vs_probe']:+.4f}     {r['margin_retr_vs_random']:+.4f}      {r['margin_random_vs_probe']:+.4f}     | {share:.1%}")

def agg(xs):
    xs = np.array(xs, dtype=float)
    out = {'mean': round(float(xs.mean()), 4), 'std': round(float(xs.std(ddof=1)), 4) if len(xs) > 1 else None}
    return out

def margin_stats(key):
    m = np.array([r[key] for r in results], dtype=float)
    out = {**agg(m), 'positive': int((m > 0).sum()), 'n': len(m)}
    if len(m) >= 2 and m.std(ddof=1) > 0:
        half = stats.t.ppf(0.975, len(m) - 1) * m.std(ddof=1) / np.sqrt(len(m))
        out.update({'ci95': [round(float(m.mean() - half), 4), round(float(m.mean() + half), 4)],
                    'p_ttest': round(float(stats.ttest_1samp(m, 0.0).pvalue), 4)})
    return out

summary = {
    'arms_test': {arm: agg([r[f'{arm}_test'] for r in results]) for arm in ARMS},
    'arms_val': {arm: agg([r[f'{arm}_val'] for r in results]) for arm in ARMS},
    'margins_test': {k: margin_stats(k) for k in ('margin_retr_vs_probe', 'margin_retr_vs_random', 'margin_random_vs_probe')},
    'retrieval_p_at_5': {s: round(p_at_k(NB_RETR[s], LBL[s]), 4) for s in ('val', 'test')},
    'image_share_retrieval': agg([r['retrieval_shift_img'] / r['retrieval_shift_full'] for r in results if r['retrieval_shift_full']]),
    'leakage_rate': round(leakage_rate, 4),
    'any_selected_run_hit_cap': any(r[f'{a}_hit_cap'] for r in results for a in ARMS),
    'vit_reference_not_paired': VIT_REFERENCE,
    'per_seed': results,
}
print()
for arm in ARMS:
    print(f"{arm:15s} TEST {summary['arms_test'][arm]['mean']:.4f} +- {summary['arms_test'][arm]['std']}")
for k, v in summary['margins_test'].items():
    print(f"{k:24s} {v['mean']:+.4f} +- {v['std']}  CI {v.get('ci95')}  p={v.get('p_ttest')}  positive {v['positive']}/{v['n']}")
if summary['any_selected_run_hit_cap']:
    print('\nWARNING: a selected run hit the epoch cap -- it may be under-trained; raise MAX_EPOCHS before interpreting.')

rp, rr = summary['margins_test']['margin_retr_vs_probe'], summary['margins_test']['margin_retr_vs_random']
def excludes_zero_above(v):
    return 'ci95' in v and v['ci95'][0] > 0
if excludes_zero_above(rp) and excludes_zero_above(rr):
    verdict = 'Better retrieval helps: retrieval beats both the no-retrieval baseline and the random-neighbour control.'
elif excludes_zero_above(rp):
    verdict = 'Retrieval beats the baseline, but not clearly the random-neighbour control: the gain may come from the extra parameters, not from retrieval.'
else:
    verdict = 'No reliable gain from retrieval even with the better encoder.'
summary['verdict'] = verdict
print('\nVERDICT:', verdict)

with open(os.path.join(WORK_DIR, 'd9_results.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print('RESULT_JSON:', json.dumps(summary, default=float))

## 9. Step 0: does the benefit of retrieval vary with reliability?

Per-query outcome: mean binary cross-entropy over the 14 labels, averaged over seeds. `gain_vs_probe` = loss(probe) − loss(retrieval), positive = retrieval helps that query. `gain_vs_random` = loss(random) − loss(retrieval), positive = the *retrieved* neighbours beat random ones for that query. Bands are quintiles of each feature, cut on **val** and applied to test.

In [ ]:
from scipy.stats import spearmanr

def bce(p, y, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean(1)   # per query

def seed_mean_loss(arm, split):
    return np.mean([bce(PROBS[arm][sd][split], LBL[split]) for sd in SEEDS], axis=0)

def seed_mean_probs(arm, split):
    return np.mean([PROBS[arm][sd][split] for sd in SEEDS], axis=0)

for s in ('val', 'test'):   # probe-derived features (inference-time: the probe is part of the system)
    pp = seed_mean_probs('probe', s)
    nb_mean = LBL['train'][NB_RETR[s]].mean(1)
    FEATS[s]['probe_nb_agree'] = 1 - np.abs(pp - nb_mean).mean(1)
    q = np.clip(pp, 1e-6, 1 - 1e-6)
    FEATS[s]['probe_entropy'] = -(q * np.log(q) + (1 - q) * np.log(1 - q)).mean(1)
FEATURES = list(FEATS['val'])

LOSS = {arm: {s: seed_mean_loss(arm, s) for s in ('val', 'test')} for arm in ARMS}
GAIN = {'vs_probe': {s: LOSS['probe'][s] - LOSS['retrieval'][s] for s in ('val', 'test')},
        'vs_random': {s: LOSS['random_control'][s] - LOSS['retrieval'][s] for s in ('val', 'test')}}

def boot_ci(x, n=N_BOOT, seed=42):
    rng = np.random.RandomState(seed)
    m = np.array([x[rng.randint(0, len(x), len(x))].mean() for _ in range(n)])
    return [round(float(np.percentile(m, 2.5)), 5), round(float(np.percentile(m, 97.5)), 5)]

def band_ids(feature):
    # unique edges: tied features (e.g. nb_consensus = 1.0 for many queries) get fewer, non-empty bands
    edges = np.unique(np.quantile(FEATS['val'][feature], np.linspace(0, 1, N_BANDS + 1)[1:-1]))
    return {s: np.digitize(FEATS[s][feature], edges) for s in ('val', 'test')}, len(edges) + 1

hetero = {}
for gname, g in GAIN.items():
    print(f'\n===== gain {gname} (per-query loss reduction; positive = retrieval better) =====')
    print(f"overall: val {g['val'].mean():+.5f}   test {g['test'].mean():+.5f} CI {boot_ci(g['test'])}")
    hetero[gname] = {'overall_test': round(float(g['test'].mean()), 5), 'overall_test_ci': boot_ci(g['test'])}
    for f in FEATURES:
        rv, pv = spearmanr(FEATS['val'][f], g['val'])
        rt, pt = spearmanr(FEATS['test'][f], g['test'])
        b, n_bands = band_ids(f)
        bands = []
        for k in range(n_bands):
            m = b['test'] == k
            if m.sum() < 2 or (b['val'] == k).sum() < 2:
                continue
            bands.append({'band': k, 'n': int(m.sum()), 'val_mean': round(float(g['val'][b['val'] == k].mean()), 5),
                          'test_mean': round(float(g['test'][m].mean()), 5), 'test_ci': boot_ci(g['test'][m])})
        hetero[gname][f] = {'spearman_val': round(float(rv), 4), 'p_val': round(float(pv), 4),
                            'spearman_test': round(float(rt), 4), 'p_test': round(float(pt), 4), 'bands': bands}
        print(f'{f:15s} spearman val {rv:+.3f} (p={pv:.3f})  test {rt:+.3f} (p={pt:.3f})   test by band (low->high): '
              + '  '.join(f"{x['test_mean']:+.4f}" for x in bands))

## 10. Upper bound and a realistic gate

- **oracle**: per query, use whichever of probe / retrieval has the lower loss on that query's *true* labels. Not achievable; it is the ceiling for any gate that routes between these two models.
- **average**: mean of the two models' probabilities. Any combination of two models can gain from plain ensembling, so a gate must beat this, not only the probe.
- **routed by feature**: use the retrieval model when the feature is above (or below) a threshold, else the probe. Threshold (val deciles) and direction are chosen on val macro AUROC, then applied once to test. Routing between two *trained* models avoids D7's problem (zeroed evidence the model never saw in training).

All numbers are test macro AUROC, computed per seed and averaged; differences are paired by seed.

In [ ]:
def per_seed_auroc(make_probs, split='test'):
    return np.array([macro_auroc(LBL[split], make_probs(sd, split)) for sd in SEEDS])

def arm_probs(arm):
    return lambda sd, s: PROBS[arm][sd][s]

def oracle(sd, s):
    lp, lr = bce(PROBS['probe'][sd][s], LBL[s]), bce(PROBS['retrieval'][sd][s], LBL[s])
    return np.where((lr < lp)[:, None], PROBS['retrieval'][sd][s], PROBS['probe'][sd][s])

def average(sd, s):
    return (PROBS['probe'][sd][s] + PROBS['retrieval'][sd][s]) / 2

def routed(feature, thr, direction):
    def f(sd, s):
        use_r = FEATS[s][feature] > thr if direction == 'above' else FEATS[s][feature] <= thr
        return np.where(use_r[:, None], PROBS['retrieval'][sd][s], PROBS['probe'][sd][s])
    return f

rows = {name: per_seed_auroc(fn) for name, fn in
        [('probe', arm_probs('probe')), ('retrieval', arm_probs('retrieval')),
         ('random_control', arm_probs('random_control')), ('average', average), ('oracle', oracle)]}
gates = {}
for f in FEATURES:
    best = None
    for thr in np.quantile(FEATS['val'][f], np.linspace(0.1, 0.9, 9)):
        for direction in ('above', 'below'):
            v = per_seed_auroc(routed(f, thr, direction), 'val').mean()
            if best is None or v > best[0]:
                best = (v, thr, direction)
    rows[f'routed:{f}'] = per_seed_auroc(routed(f, best[1], best[2]))
    gates[f] = {'val_auroc': round(float(best[0]), 4), 'threshold': round(float(best[1]), 4), 'use_retrieval_when': best[2],
                'share_routed_to_retrieval_test': round(float(np.mean(FEATS['test'][f] > best[1] if best[2] == 'above'
                                                                       else FEATS['test'][f] <= best[1])), 3)}

def paired(a, b):
    d = rows[a] - rows[b]
    p = float(stats.ttest_1samp(d, 0).pvalue) if d.std() > 0 else 1.0
    return {'mean': round(float(d.mean()), 4), 'positive': int((d > 0).sum()), 'p': round(p, 4)}

print(f"{'model':28s} test macro AUROC (mean over {len(SEEDS)} seeds)   vs probe   vs average")
gate_table = {}
for name, v in rows.items():
    gate_table[name] = {'test_mean': round(float(v.mean()), 4), 'test_std': round(float(v.std(ddof=1)), 4),
                        'vs_probe': paired(name, 'probe') if name != 'probe' else None,
                        'vs_average': paired(name, 'average') if name not in ('average', 'probe') else None}
    vp, va = gate_table[name]['vs_probe'], gate_table[name]['vs_average']
    vp_s = '' if vp is None else f"{vp['mean']:+.4f} (p={vp['p']:.2f})"
    va_s = '' if va is None else f"{va['mean']:+.4f} (p={va['p']:.2f})"
    print(f"{name:28s} {v.mean():.4f} +- {v.std(ddof=1):.4f}   {vp_s:18s} {va_s}")

## 11. Calibration (contribution 4): expected calibration error and Brier score, per arm, test

In [ ]:
def ece(p, y, bins=10):
    out = []
    for c in range(y.shape[1]):
        idx = np.clip((p[:, c] * bins).astype(int), 0, bins - 1)
        e = sum(abs(p[idx == b, c].mean() - y[idx == b, c].mean()) * (idx == b).mean()
                for b in range(bins) if (idx == b).any())
        out.append(e)
    return float(np.mean(out))

calib = {}
for name, fn in [('probe', arm_probs('probe')), ('retrieval', arm_probs('retrieval')),
                 ('random_control', arm_probs('random_control')), ('average', average)]:
    e = np.array([ece(fn(sd, 'test'), LBL['test']) for sd in SEEDS])
    br = np.array([((fn(sd, 'test') - LBL['test']) ** 2).mean() for sd in SEEDS])
    calib[name] = {'ece_mean': round(float(e.mean()), 4), 'ece_std': round(float(e.std(ddof=1)), 4),
                   'brier_mean': round(float(br.mean()), 4)}
    print(f"{name:15s} ECE {e.mean():.4f} +- {e.std(ddof=1):.4f}   Brier {br.mean():.4f}")

## 12. Verdict and outputs

In [ ]:
best_gate = max((k for k in rows if k.startswith('routed:')), key=lambda k: rows[k].mean())
headroom = gate_table['oracle']['vs_probe']['mean']
monotone = {f: hetero['vs_random'][f]['spearman_test'] for f in FEATURES}
strong = [f for f in FEATURES if hetero['vs_random'][f]['p_val'] < 0.05 and hetero['vs_random'][f]['p_test'] < 0.05
          and np.sign(hetero['vs_random'][f]['spearman_val']) == np.sign(hetero['vs_random'][f]['spearman_test'])]
gate_beats_avg = gate_table[best_gate]['vs_average']['mean'] > 0 and gate_table[best_gate]['vs_average']['p'] < 0.05
if strong and gate_beats_avg:
    verdict = (f'GO: the advantage of retrieved over random neighbours varies with {strong} on both val and test, and '
               f'{best_gate} beats simple averaging. A reliability-aware design has something real to exploit.')
elif strong:
    verdict = (f'WEAK GO: retrieval quality matters per query ({strong}), but no simple gate beats averaging the two '
               f'models. A learned, train-time gate might; a threshold does not.')
else:
    verdict = ('NO-GO: whether retrieved neighbours beat random ones does not depend on any reliability feature '
               'consistently on val and test. A reliability gate would have nothing retrieval-specific to exploit.')
print('VERDICT:', verdict)
print(f'oracle headroom over probe: {headroom:+.4f} AUROC (ceiling, not achievable)')

np.savez_compressed(os.path.join(WORK_DIR, 'reliability_step0_outputs.npz'),
                    **{f'{arm}_seed{sd}_{s}': PROBS[arm][sd][s] for arm in ARMS for sd in SEEDS for s in ('val', 'test')},
                    **{f'feat_{f}_{s}': FEATS[s][f] for f in FEATURES for s in ('val', 'test')},
                    **{f'{s}_lbl': LBL[s] for s in ('val', 'test')}, **{f'{s}_pid': PIDS[s] for s in ('val', 'test')},
                    **{f'{s}_nb': NB_RETR[s] for s in ('val', 'test')})
summary = {'d9_replication': {arm: round(float(np.mean([r[f'{arm}_test'] for r in results])), 4) for arm in ARMS},
           'heterogeneity': hetero, 'gates': gates, 'gate_table': gate_table, 'calibration': calib,
           'oracle_headroom_vs_probe': headroom, 'best_gate': best_gate, 'verdict': verdict}
with open(os.path.join(WORK_DIR, 'reliability_step0_results.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print('RESULT_JSON:', json.dumps(summary, default=float))

## How to read this

- **Main question: does `gain_vs_random` vary with a reliability feature?** That is the retrieval-specific effect. A significant Spearman correlation in the same direction on val and test, with bands rising steadily, means retrieved neighbours help when they are reliable and not otherwise.
- **`gain_vs_probe`** mixes in the extra parameters of the evidence pathway, so on its own it can't show that reliability matters.
- **Oracle headroom** is the most any probe/retrieval router could gain. A small headroom (under ~0.01) makes the whole direction hard to justify.
- **A gate must beat `average`**, not only `probe`. Two models averaged usually beat either one.
- Bands have ~150 test queries each, so per-band numbers are noisy. Trust the val-and-test agreement more than any single band.